# Restrat Session after run first cell

In [1]:
# === 1. Comprehensive Library Installation (CUDA 12.8 Compatible) ===
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --force-reinstall -q
!pip install --upgrade transformers accelerate bitsandbytes Pillow requests apify-client sentencepiece pandas -q

# === 2. Verifying Installation ===
import torch, bitsandbytes as bnb
print(f" PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f" GPU: {torch.cuda.get_device_name(0)}")
print(f" bitsandbytes: {bnb.__version__}")
print("\n Installation complete! If RAM is still full, click Restart from the top menu so the updates take effect in Python.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 20.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 88.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 80.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 113.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 237.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 117.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 221.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 220.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 118.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 104.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 80.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

# Restart and run from here

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_DIR = "/content/drive/MyDrive/hajj_real_data"
os.makedirs(DRIVE_DIR, exist_ok=True)

IMAGES_DIR = os.path.join(DRIVE_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

CSV_PATH = os.path.join(DRIVE_DIR, "real_data.csv")

print(f"[OK] drive connected")
print(f"[OK] csv path: {CSV_PATH}")
print(f"[OK] images path: {IMAGES_DIR}")


Mounted at /content/drive
[OK] drive connected
[OK] csv path: /content/drive/MyDrive/hajj_real_data/real_data.csv
[OK] images path: /content/drive/MyDrive/hajj_real_data/images


In [2]:
# Ensure torch + torchvision are working together
import torch, torchvision
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print(f" PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f" torchvision: {torchvision.__version__}")

# Ensure transformers + LLaVA are working
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, pipeline
print(" LlavaForConditionalGeneration loaded!")
import torch
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f" bitsandbytes: {bnb.__version__}")

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
print(" BitsAndBytesConfig is ready!")
print(" Everything is set - Start loading the model!")


 PyTorch: 2.11.0+cu128 | CUDA: 12.8
 torchvision: 0.26.0+cu128
 LlavaForConditionalGeneration loaded!
CUDA: 12.8
GPU: Tesla T4
 bitsandbytes: 0.49.2
 BitsAndBytesConfig is ready!
 Everything is set - Start loading the model!


In [3]:
import gc, torch, os
from PIL import Image as PILImage
from transformers import LlavaForConditionalGeneration, AutoProcessor, pipeline, BitsAndBytesConfig
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(" (1/2) Loading LLaVA 1.5 7B 4bit...")
processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print(" LLaVA ready in 4-bit!")

print(" (2/2) Loading NLLB for translation...")


 GPU: Tesla T4 | CUDA: 12.8
 (1/2) Loading LLaVA 1.5 7B 4bit...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

 LLaVA ready in 4-bit!
 (2/2) Loading NLLB for translation...


In [11]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from PIL import Image as PILImage
nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model_trans = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M").to(device)
print(" NLLB ready and no memory issues!")

print(" Ready - Run the Script Cell!")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

 NLLB ready and no memory issues!
 Ready - Run the Script Cell!


In [6]:
# Install arabert if not already installed
!pip install arabert -q

import os
import torch
import joblib
import pandas as pd
import numpy as np
import re
import nltk
from PIL import Image as PILImage
from transformers import (
    AutoProcessor, LlavaForConditionalGeneration,
    AutoModelForSeq2SeqLM, AutoTokenizer,
    AutoModelForSequenceClassification, BitsAndBytesConfig
)
from arabert.preprocess import ArabertPreprocessor
from nltk.stem.isri import ISRIStemmer

# Mount Drive first if not already done
# from google.colab import drive
# drive.mount('/content/drive')

# Define Paths
ARABERT_PATH = "/content/drive/MyDrive/hajj_processing_workspace/arabert_fine_tuning"
SVM_PATH = "/content/drive/MyDrive/hajj_processing_workspace/svm97/svm97_model.pkl"

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 12.2 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Device: cuda


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
# 1. Load AraBERT V2
print("Loading AraBERT V2 Classifier...")
bert_tokenizer = AutoTokenizer.from_pretrained(ARABERT_PATH)
bert_model = AutoModelForSequenceClassification.from_pretrained(ARABERT_PATH).to(device)
bert_preprocessor = ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02")

# 2. Load SVM Pipeline & Label Encoder
print("Loading SVM Pipeline...")
svm_pipeline = joblib.load(SVM_PATH)
le = joblib.load(f"{ARABERT_PATH}/label_encoder.pkl")

st = ISRIStemmer()

print("All classifiers loaded.")

Loading AraBERT V2 Classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading SVM Pipeline...
All classifiers loaded.


In [8]:
def translate_to_arabic(text):
    inputs = nllb_tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = nllb_model_trans.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("arb_Arab"),
        max_new_tokens=300
    )
    return nllb_tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

def get_image_analysis(image_path):
    prompt = "USER: <image>\nAnalyze this Hajj advertisement. What is sold? List prices, dates, company name, and phone numbers if visible. ASSISTANT:"
    img = PILImage.open(image_path).convert("RGB")
    inputs = processor(text=prompt, images=img, return_tensors="pt").to("cuda")
    output = llava_model.generate(**inputs, max_new_tokens=250)
    en_desc = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
    return translate_to_arabic(en_desc)

def svm_clean_stem(text):
    text = re.sub(r'[^\w\s]', '', str(text))
    tokens = nltk.word_tokenize(text)
    return " ".join([st.stem(w) for w in tokens])

In [13]:
def run_full_ad_analysis(image_path, post_text):
    print("--- Phase 1: Analyzing Image (LLaVA) ---")
    img_arabic_desc = get_image_analysis(image_path)
    print(f"Image Desc: {img_arabic_desc}")

    full_context = post_text + " " + img_arabic_desc

    # --- AraBERT Prediction (Works fine with Probabilities) ---
    clean_bert = bert_preprocessor.preprocess(full_context)
    bert_inputs = bert_tokenizer(clean_bert, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    bert_model.eval()
    with torch.no_grad():
        bert_logits = bert_model(**bert_inputs).logits
        bert_pred_id = torch.argmax(bert_logits, dim=1).item()
        bert_probs = torch.nn.functional.softmax(bert_logits, dim=1)

    bert_label = le.inverse_transform([bert_pred_id])[0]
    bert_score = bert_probs[0][bert_pred_id].item() * 100

    stemmed_text = svm_clean_stem(full_context)
    svm_pred = svm_pipeline.predict([stemmed_text])[0]

    svm_score_display = "N/A (Prob=False)"

    try:
        decision_val = svm_pipeline.decision_function([stemmed_text])[0]
        svm_score_display = f"Dist: {abs(decision_val):.2f}"
    except:
        pass

    print("\n" + "="*40)
    print(f"FINAL REPORT")
    print(f"AraBERT V2: {bert_label} ({bert_score:.2f}%)")
    print(f"SVM Model: {svm_pred} ({svm_score_display})")
    print("="*40)

    return img_arabic_desc

In [14]:
# SET YOUR INPUTS HERE
TEST_IMAGE = "/content/drive/MyDrive/hajj_processing_workspace/download111111111111.jpeg"
TEST_POST = """نفسك تطلع الحج؟ حاولت كام مره تطلع التأشيره وفشلت؟ دلوقتي مع العمده ترافيل هتحقق حلمك استخراج تأشيره في 20 يوم فقط ورحله الحج كامله بمبلغ 50 الف جنيه فقط !!!! ايوه زي ما سمعت 50 الف في برج الساعه تواصل واتس لسرعه الحجز او بادر بتحويل جزء من المبلغ عبر محفظه فودافون كاش 01035377752
اب"""

# Run the pipeline
analysis = run_full_ad_analysis(TEST_IMAGE, TEST_POST)

--- Phase 1: Analyzing Image (LLaVA) ---


Both `max_new_tokens` (=300) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Image Desc: يروج الإعلان لحج الحج إلى مكة، السعودية، وتظهر الصورة مسجدة كبيرة ومضاءة جيداً مع حشد من الناس المتجمعين حول الكعبه، وهي الهيكل المركزي للمسجد، ويُكتب الإعلان باللغة العربية، ولا يوجد معلومات واضحة عن الأسعار، ومع ذلك، يتم توفير اسم الشركة ورقم الهاتف للمهاجرين المحتملين للتواصل لمزيد من المعلومات.

FINAL REPORT
AraBERT V2: Suspicious (84.02%)
SVM Model: Suspicious (Dist: 0.89)
